# ZAIC DataMate — Kaggle GUI migration

Bản chuẩn bị ngày 07/11/2026. Không cần Kaggle CLI. Notebook này chưa được xác nhận chạy GPU Kaggle.

**Lượt đầu chỉ preflight.** Mã v2.1 không đổi. Không mount Drive; không tải model/API từ mạng; không nộp bài. Input base/SFT phải do người dùng gắn bằng GUI và được phép theo thể lệ. Private không tự chứng minh được phép xử lý dữ liệu cuộc thi trên Kaggle.

Đọc tài liệu bàn giao trước. Không dùng notebook Colab lịch sử với Run All.


## 1. Cấu hình

Dùng **Add Input / Upload** để gắn ZIP mã bàn giao, rồi copy đường dẫn ZIP vào CODE_ZIP. Đây là thao tác Python trong notebook, không phải lệnh CLI. Các đường dẫn mẫu dưới đây cần thay theo tên input thực tế.

- preflight: chỉ code + 99 tests, CPU được.
- smoke: cần base input và GPU; dữ liệu giả lập, không train adapter lưu lại.
- train: cần quyền cloud-use, frozen SFT + manifest; train fresh adapter từ base, không resume adapter cũ.


In [ ]:
CODE_ZIP = "/kaggle/input/REPLACE_WITH_CODE_DATASET/ZAIC_Local_Codex_Kaggle_GUI_20261107.zip"
CONFIG = {
    "ACTION": "preflight",  # preflight | smoke | train
    "BASE_INPUT": "/kaggle/input/REPLACE_WITH_BASE_DATASET/base",
    "SFT_INPUT": "/kaggle/input/REPLACE_WITH_APPROVED_SFT_DATASET/data",
    "GPU_RUN_CONFIRMED": False,
    "COMPETITION_CLOUD_USE_ALLOWED": False,
    "TRAIN_FRESH_ADAPTER": False,
}


## 2. Giải nén mã và chạy tác vụ

Cell này không tự cài thư viện. Nếu môi trường không tương thích, đưa báo cáo versions/lỗi đã lọc cho Codex để chuẩn bị cell môi trường phù hợp. Không cài đè torch/cu130 của Colab một cách máy móc.

Nếu ACTION=train, mỗi lần chạy tạo một experiment mới. Không bấm chạy lại để mong resume optimizer. Giữ notebook và output Private.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
from datetime import datetime, timezone
import importlib.util, json, stat

archive = Path(CODE_ZIP)
if not archive.is_file() or not str(archive.resolve()).startswith('/kaggle/input/'):
    raise ValueError("Gắn ZIP mã bằng Add Input rồi điền CODE_ZIP theo đường dẫn thực.")
extract_root = Path('/kaggle/working/zaic_gui_code') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
extract_root.mkdir(parents=True, exist_ok=False)
with ZipFile(archive) as z:
    if sum(info.file_size for info in z.infolist()) > 25 * 1024 * 1024:
        raise ValueError("ZIP mã quá lớn; không chọn ZIP dữ liệu/weights.")
    for info in z.infolist():
        target = (extract_root / info.filename).resolve()
        if not target.is_relative_to(extract_root.resolve()) or stat.S_ISLNK(info.external_attr >> 16):
            raise ValueError("Unsafe ZIP entry")
    z.extractall(extract_root)
projects = list(extract_root.rglob('CODEX_PROJECT/gui/kaggle_runner.py'))
if len(projects) != 1:
    raise ValueError("Chọn đúng ZIP bàn giao có CODEX_PROJECT/gui/kaggle_runner.py")
source_root = projects[0].parents[1]
spec = importlib.util.spec_from_file_location('zaic_kaggle_gui_runner', projects[0])
runner = importlib.util.module_from_spec(spec)
spec.loader.exec_module(runner)
RESULT = runner.run(CONFIG, source_root)
print(json.dumps({"state": RESULT['state'], "training_started": CONFIG['ACTION'] == 'train'}, indent=2))


## 3. Lưu phiên bản và tải kết quả

Khi preflight đạt, bật GPU ở Session options và chạy smoke. Chỉ sau khi quyền xử lý dữ liệu được xác minh và smoke đạt mới chọn train. Mỗi lượt GPU tạo thư mục `/kaggle/working/zaic_runs/<timestamp>_<action>`.

Khi train hoàn tất, kiểm `training_complete.json`, `GUI_RUN_METADATA.json`, adapter SHA256, số steps/rows và finite gradients. Chưa đánh giá TRAIN hay PA public ở notebook này.

Dùng Save Version để lưu outputs của phiên đã chạy; đọc tùy chọn để tránh Save & Run All chạy lại huấn luyện. Tải output về máy bằng GUI và kiểm tra đủ checkpoint/adapter/metadata. Không public output. Nếu bị ngắt cứng, kiểm và loại symlink base khỏi vùng output trước lưu; runner tự gỡ link khi kết thúc bình thường hoặc báo lỗi có kiểm soát.

Không bao giờ nộp artifact chỉ vì loss đã giảm. Cần notebook đánh giá riêng trên TRAIN, batch parity và kiểm deployment offline.
